In [1]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [3]:
# Dataset statistics before tokenization
emotions_unique = df["emotion"].unique()
print("Dataset statistics before tokenization for each unique emotion:")
print()
for emotion in emotions_unique:
    subset = df[df["emotion"] == emotion]
    N = len(subset["emotion"])
    print(f"{emotion} has {N} occurences, {round(N/len(df["emotion"]) * 100, 2)}%")
    average_length = subset["content"].str.len().mean()
    print(f"{emotion} has an average length of {average_length:.2f} characters")
    print()


Dataset statistics before tokenization for each unique emotion:

joy has 1092 occurences, 14.53%
joy has an average length of 103.14 characters

fear has 1076 occurences, 14.32%
fear has an average length of 123.96 characters

anger has 1079 occurences, 14.36%
anger has an average length of 130.65 characters

sadness has 1082 occurences, 14.4%
sadness has an average length of 103.93 characters

disgust has 1066 occurences, 14.18%
disgust has an average length of 114.61 characters

shame has 1071 occurences, 14.25%
shame has an average length of 116.73 characters

guilt has 1050 occurences, 13.97%
guilt has an average length of 124.89 characters



In [4]:
# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


In [5]:
df.head(5)
# print avarage token length of sentence here.

,emotion,content,tokens
0,joy,On days when I feel close to my partner and ot...,"[On, days, when, I, feel, close, to, my, partn..."
1,fear,Every time I imagine that someone I love or I ...,"[Every, time, I, imagine, that, someone, I, lo..."
2,anger,When I had been obviously unjustly treated and...,"[When, I, had, been, obviously, unjustly, trea..."
3,sadness,When I think about the short time that we live...,"[When, I, think, about, the, short, time, that..."
4,disgust,At a gathering I found myself involuntarily si...,"[At, a, gathering, I, found, myself, involunta..."


In [6]:
# Representation 2: Bag of words
vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)

bag_matrix = vectorizer.fit_transform(df["tokens"].tolist())

bag_df = pd.DataFrame(bag_matrix.toarray(),
                      columns=vectorizer.get_feature_names_out(),
                      index=df.index
)

print(bag_df.head(5))

   !  $  %  &  '  ''  'd  'll  'm  're  ...  yr.  yrs  yukky  zero  \
0  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
1  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
2  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
3  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
4  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   

   zig-zagging  zip  zipper  zombies  zone  zoophiliac  
0            0    0       0        0     0           0  
1            0    0       0        0     0           0  
2            0    0       0        0     0           0  
3            0    0       0        0     0           0  
4            0    0       0        0     0           0  

[5 rows x 10173 columns]


In [7]:
# Representation 3: Bigrams

bigrams = []
for sentence in df["tokens"]:
    bigrams_per_sentence = []
    for i in range(len(sentence) - 1):
        bigram = sentence[i] + " " + sentence[i+1]
        bigrams_per_sentence.append(bigram)
    bigrams.append(bigrams_per_sentence)
df["bigrams"] = bigrams

In [8]:
# Representation 4: Trigrams

trigrams = []
for sentence in df["tokens"]:
    trigrams_per_sentence = []
    for i in range(len(sentence) - 2):
        trigram = sentence[i] + " " + sentence[i+1] + " " + sentence[i+2]
        trigrams_per_sentence.append(trigram)
    trigrams.append(trigrams_per_sentence)
df["trigrams"] = trigrams

In [9]:
print(df["trigrams"][0])

['On days when', 'days when I', 'when I feel', 'I feel close', 'feel close to', 'close to my', 'to my partner', 'my partner and', 'partner and other', 'and other friends', 'other friends .', 'friends . When', '. When I', 'When I feel', 'I feel at', 'feel at peace', 'at peace with', 'peace with myself', 'with myself and', 'myself and also', 'and also experience', 'also experience a', 'experience a close', 'a close contact', 'close contact with', 'contact with people', 'with people whom', 'people whom I', 'whom I regard', 'I regard greatly', 'regard greatly .']


In [10]:
# Architecture 1: Log-Linear
import itertools
nltk.download("punkt_tab")

corpus = " ".join(df["content"].fillna("").astype(str))

vocab = sorted(set(word_tokenize(corpus)))
token_to_id = {token: i for i, token in enumerate(vocab)}

class Encoder:
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# for seed in my_seeds:  
#     my_cross_validation_sets.append(
#         cross_validation_split(
#             # we pair the texts with their labels
#             pairs=list(zip(dataset["train"]["text"], dataset["train"]["label"])), 
#             k=num_folds,
#             seed=seed
#         )
#     )

class TokenIndicator(Encoder):
    def __init__(self, token_to_id):
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.token_to_id)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self), dtype=np.float32)

        for token in set(word_tokenize(x)):
            token_id = self.token_to_id.get(token)
            if token_id is not None:
                vector[token_id] = 1

        return vector
    
token_indicator_encoder = TokenIndicator(token_to_id)
doc_repr = token_indicator_encoder(corpus)

print(len(doc_repr))
print(doc_repr.sum())

[nltk_data] Downloading package punkt_tab to /home/colin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


10176
10176.0


In [11]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [12]:
# Architecture 2: Single hidden layer MLP
import smlp

class CountVectorizerEncoder(Encoder):
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer

    def __len__(self):
        return len(self.vectorizer.vocabulary_)

    def __call__(self, x: str) -> np.ndarray:
        return self.vectorizer.transform([x]).toarray().ravel()

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)

# Settings for encoders for token indicator, BOW and unigrams + trigrams (capped due to memory issues).
encoder_configs = {
    "token-indicator": {
        "binary": True,
        "ngram_range": (1, 1),
    },
    "bag-of-words": {
        "binary": False,
        "ngram_range": (1, 1),
    },
    "bigrams-and-trigrams": {
        "binary": False,
        "ngram_range": (1, 2),
        "max_features": 20000
    },
}

results = {name: [] for name in encoder_configs}
mean_accuracy = {}
# For each encoder run the model with hidden dim of 128 with cross validation.
for encoder_name, config in encoder_configs.items():
    for fold in cv:
        print(encoder_name)
        training = cv[fold]["training"]
        validation = cv[fold]["validation"]

        vectorizer = CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            **config,
        )
        print("Fitting vectorizer")
        vectorizer.fit([content for content, label in training])
        print("Vocabulary size:", len(vectorizer.vocabulary_))
        print("Training MLP")
        encoder = CountVectorizerEncoder(vectorizer)

        metrics, best_epoch, best_pred = smlp.train_neural_network_model(
            training,
            validation,
            num_classes=len(labels),
            encoder=encoder,
            save_to=f"{results_folder}/mlp_{encoder_name}_{fold}_test",
            num_epochs=10,
        )

        best_val_acc_this_fold = max(epoch_data["val_metrics"]["accuracy"] for epoch_data in metrics)
        mean_accuracy[f"{encoder_name}"] = np.mean(best_val_acc_this_fold)
print(mean_accuracy)


token-indicator
Fitting vectorizer
Vocabulary size: 8343
Training MLP
Transforming documents...
Document transformation done! Time taken: 2.819 seconds
Dataloader done! Time taken: 0.866 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

/home/colin/Downloads/nlp1-student-main/NLP-Classification-Project/Code/smlp.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/home/colin/Downloads/nlp1-student-main/NLP-Classification-Project/Code/smlp.py:71: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:916.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()


Epoch 01 | train loss: 1.7919 | train acc: 0.4182 | val loss: 1.5481 | val acc: 0.5412
Epoch 02 | train loss: 1.1690 | train acc: 0.6732 | val loss: 1.2389 | val acc: 0.5672
Epoch 03 | train loss: 0.7587 | train acc: 0.7869 | val loss: 1.1667 | val acc: 0.5871
Epoch 04 | train loss: 0.5346 | train acc: 0.8571 | val loss: 1.1831 | val acc: 0.5844
Epoch 05 | train loss: 0.3960 | train acc: 0.9007 | val loss: 1.2298 | val acc: 0.5811
Epoch 06 | train loss: 0.2980 | train acc: 0.9290 | val loss: 1.2875 | val acc: 0.5811
Epoch 07 | train loss: 0.2355 | train acc: 0.9463 | val loss: 1.3603 | val acc: 0.5738
Epoch 08 | train loss: 0.1870 | train acc: 0.9581 | val loss: 1.4315 | val acc: 0.5685
Epoch 09 | train loss: 0.1545 | train acc: 0.9679 | val loss: 1.5000 | val acc: 0.5685
Epoch 10 | train loss: 0.1236 | train acc: 0.9785 | val loss: 1.5794 | val acc: 0.5632
Training done! Time taken: 15.811 seconds
token-indicator
Fitting vectorizer
Vocabulary size: 8399
Training MLP
Transforming docum

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7969 | train acc: 0.3973 | val loss: 1.5529 | val acc: 0.5150
Epoch 02 | train loss: 1.1779 | train acc: 0.6715 | val loss: 1.2253 | val acc: 0.5802
Epoch 03 | train loss: 0.7589 | train acc: 0.7920 | val loss: 1.1616 | val acc: 0.5882
Epoch 04 | train loss: 0.5344 | train acc: 0.8603 | val loss: 1.1799 | val acc: 0.5822
Epoch 05 | train loss: 0.3982 | train acc: 0.9022 | val loss: 1.2223 | val acc: 0.5848
Epoch 06 | train loss: 0.3028 | train acc: 0.9285 | val loss: 1.2854 | val acc: 0.5775
Epoch 07 | train loss: 0.2387 | train acc: 0.9476 | val loss: 1.3367 | val acc: 0.5709
Epoch 08 | train loss: 0.1920 | train acc: 0.9579 | val loss: 1.3977 | val acc: 0.5742
Epoch 09 | train loss: 0.1567 | train acc: 0.9662 | val loss: 1.4631 | val acc: 0.5689
Epoch 10 | train loss: 0.1312 | train acc: 0.9739 | val loss: 1.5262 | val acc: 0.5695
Training done! Time taken: 14.207 seconds
token-indicator
Fitting vectorizer
Vocabulary size: 8413
Training MLP
Transforming docum

... (22 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6981 | train acc: 0.4242 | val loss: 1.3847 | val acc: 0.5742
Epoch 02 | train loss: 0.8523 | train acc: 0.7805 | val loss: 1.1574 | val acc: 0.5921
Epoch 03 | train loss: 0.3886 | train acc: 0.9183 | val loss: 1.1574 | val acc: 0.6021
Epoch 04 | train loss: 0.1948 | train acc: 0.9682 | val loss: 1.2064 | val acc: 0.5975
Epoch 05 | train loss: 0.1088 | train acc: 0.9857 | val loss: 1.2809 | val acc: 0.5882
Epoch 06 | train loss: 0.0674 | train acc: 0.9952 | val loss: 1.3451 | val acc: 0.5775
Epoch 07 | train loss: 0.0451 | train acc: 0.9962 | val loss: 1.4064 | val acc: 0.5775
Epoch 08 | train loss: 0.0334 | train acc: 0.9968 | val loss: 1.4708 | val acc: 0.5702
Epoch 09 | train loss: 0.0264 | train acc: 0.9973 | val loss: 1.5193 | val acc: 0.5729
Epoch 10 | train loss: 0.0206 | train acc: 0.9978 | val loss: 1.5736 | val acc: 0.5689
Training done! Time taken: 27.000 seconds
bigrams-and-trigrams
Fitting vectorizer
Vocabulary size: 20000
Training MLP
Transforming

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6946 | train acc: 0.4329 | val loss: 1.3580 | val acc: 0.5709
Epoch 02 | train loss: 0.8435 | train acc: 0.7798 | val loss: 1.1416 | val acc: 0.5981
Epoch 03 | train loss: 0.3840 | train acc: 0.9178 | val loss: 1.1241 | val acc: 0.6154
Epoch 04 | train loss: 0.1924 | train acc: 0.9696 | val loss: 1.1743 | val acc: 0.6108
Epoch 05 | train loss: 0.1111 | train acc: 0.9854 | val loss: 1.2313 | val acc: 0.6028
Epoch 06 | train loss: 0.0694 | train acc: 0.9928 | val loss: 1.2933 | val acc: 0.5981
Epoch 07 | train loss: 0.0491 | train acc: 0.9953 | val loss: 1.3367 | val acc: 0.5961
Epoch 08 | train loss: 0.0350 | train acc: 0.9963 | val loss: 1.3901 | val acc: 0.5915
Epoch 09 | train loss: 0.0276 | train acc: 0.9970 | val loss: 1.4263 | val acc: 0.5915
Epoch 10 | train loss: 0.0226 | train acc: 0.9970 | val loss: 1.4766 | val acc: 0.5882
Training done! Time taken: 23.922 seconds
{'token-indicator': np.float64(0.6154357950765137), 'bag-of-words': np.float64(0.6101131

In [ ]:
# Architecture 3: Modified MLP - 2 hidden layers

import smlp_mod
class CountVectorizerEncoder(Encoder):
    def __init__(self, vectorizer):
        self.vectorizer = vectorizer

    def __len__(self):
        return len(self.vectorizer.vocabulary_)

    def __call__(self, x: str) -> np.ndarray:
        return self.vectorizer.transform([x]).toarray().ravel()

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)

encoder_configs = {
    "token-indicator": {
        "binary": True,
        "ngram_range": (1, 1),
    },
    "bag-of-words": {
        "binary": False,
        "ngram_range": (1, 1),
    },
    "bigrams-and-trigrams": {
        "binary": False,
        "ngram_range": (1, 2),
        "max_features": 75000
    },
}

results = {name: [] for name in encoder_configs}

encoder_accuracy = {}
for encoder_name, config in encoder_configs.items():
    mean_accuracy = []
    for fold in cv:
        print(encoder_name)
        training = cv[fold]["training"]
        validation = cv[fold]["validation"]

        vectorizer = CountVectorizer(
            tokenizer=word_tokenize,
            token_pattern=None,
            **config,
        )
        vectorizer.fit([content for content, label in training])
        print("Vocabulary size:", len(vectorizer.vocabulary_))
        encoder = CountVectorizerEncoder(vectorizer)

        metrics, best_epoch, best_pred = smlp_mod.train_neural_network_model(
            training,
            validation,
            num_classes=len(labels),
            encoder=encoder,
            save_to=f"{results_folder}/mlp_{encoder_name}_{fold}_test",
            num_epochs=10,
        )

        results[encoder_name].append(
            {
                "metrics": metrics,
                "best_epoch": best_epoch,
                "best_pred": best_pred,
            }
        )

        best_val_acc_this_fold = max(epoch_data["val_metrics"]["accuracy"] for epoch_data in metrics)
        mean_accuracy.append(best_val_acc_this_fold)
    encoder_accuracy[f"{encoder_name}"] = np.mean(mean_accuracy)



token-indicator
Vocabulary size: 8343
Transforming documents...
Document transformation done! Time taken: 2.682 seconds
Dataloader done! Time taken: 0.323 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6594 | train acc: 0.3576 | val loss: 1.3549 | val acc: 0.5306
Epoch 02 | train loss: 1.1529 | train acc: 0.5936 | val loss: 1.2322 | val acc: 0.5592
Epoch 03 | train loss: 0.8986 | train acc: 0.7013 | val loss: 1.2555 | val acc: 0.5459
Epoch 04 | train loss: 0.7678 | train acc: 0.7460 | val loss: 1.2683 | val acc: 0.5585
Epoch 05 | train loss: 0.6367 | train acc: 0.7864 | val loss: 1.3101 | val acc: 0.5559
Epoch 06 | train loss: 0.5750 | train acc: 0.8139 | val loss: 1.3150 | val acc: 0.5545
Epoch 07 | train loss: 0.5330 | train acc: 0.8287 | val loss: 1.3719 | val acc: 0.5465
Epoch 08 | train loss: 0.5012 | train acc: 0.8332 | val loss: 1.4000 | val acc: 0.5532
Epoch 09 | train loss: 0.4573 | train acc: 0.8566 | val loss: 1.4542 | val acc: 0.5512
Epoch 10 | train loss: 0.4445 | train acc: 0.8588 | val loss: 1.5126 | val acc: 0.5532
Training done! Time taken: 11.566 seconds
token-indicator
Vocabulary size: 8399
Transforming documents...
Document transformation 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.6641 | train acc: 0.3501 | val loss: 1.3088 | val acc: 0.5403
Epoch 02 | train loss: 1.1838 | train acc: 0.5809 | val loss: 1.2499 | val acc: 0.5509
Epoch 03 | train loss: 0.9507 | train acc: 0.6659 | val loss: 1.2300 | val acc: 0.5635
Epoch 04 | train loss: 0.7897 | train acc: 0.7243 | val loss: 1.2879 | val acc: 0.5516
Epoch 05 | train loss: 0.6704 | train acc: 0.7773 | val loss: 1.3209 | val acc: 0.5549
Epoch 06 | train loss: 0.6352 | train acc: 0.7926 | val loss: 1.3483 | val acc: 0.5562
Epoch 07 | train loss: 0.5397 | train acc: 0.8211 | val loss: 1.3957 | val acc: 0.5542
Epoch 08 | train loss: 0.5006 | train acc: 0.8349 | val loss: 1.4631 | val acc: 0.5576
Epoch 09 | train loss: 0.4697 | train acc: 0.8480 | val loss: 1.5256 | val acc: 0.5509
Epoch 10 | train loss: 0.4389 | train acc: 0.8601 | val loss: 1.4952 | val acc: 0.5629
Training done! Time taken: 11.342 seconds
token-indicator
Vocabulary size: 8413
Transforming documents...
Document transformation 

  0%|          | 0/10 [00:00<?, ?it/s]

... (11 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7315 | train acc: 0.3280 | val loss: 1.4140 | val acc: 0.4877
Epoch 02 | train loss: 1.2746 | train acc: 0.5427 | val loss: 1.2692 | val acc: 0.5482
Epoch 03 | train loss: 1.0126 | train acc: 0.6453 | val loss: 1.2637 | val acc: 0.5509
Epoch 04 | train loss: 0.8835 | train acc: 0.6945 | val loss: 1.2681 | val acc: 0.5589
Epoch 05 | train loss: 0.7486 | train acc: 0.7446 | val loss: 1.2533 | val acc: 0.5682
Epoch 06 | train loss: 0.6937 | train acc: 0.7722 | val loss: 1.2919 | val acc: 0.5595
Epoch 07 | train loss: 0.6134 | train acc: 0.7931 | val loss: 1.3940 | val acc: 0.5469
Epoch 08 | train loss: 0.5957 | train acc: 0.8082 | val loss: 1.3703 | val acc: 0.5469
Epoch 09 | train loss: 0.5414 | train acc: 0.8222 | val loss: 1.4503 | val acc: 0.5529
Epoch 10 | train loss: 0.5229 | train acc: 0.8329 | val loss: 1.4501 | val acc: 0.5316
Training done! Time taken: 11.684 seconds
bag-of-words
Vocabulary size: 8353
Transforming documents...
Document transformation don

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7114 | train acc: 0.3393 | val loss: 1.3173 | val acc: 0.5549
Epoch 02 | train loss: 1.2400 | train acc: 0.5671 | val loss: 1.2577 | val acc: 0.5615
Epoch 03 | train loss: 0.9926 | train acc: 0.6619 | val loss: 1.2379 | val acc: 0.5695
Epoch 04 | train loss: 0.8507 | train acc: 0.7121 | val loss: 1.2214 | val acc: 0.5695
Epoch 05 | train loss: 0.7464 | train acc: 0.7557 | val loss: 1.2371 | val acc: 0.5649
Epoch 06 | train loss: 0.6581 | train acc: 0.7821 | val loss: 1.3152 | val acc: 0.5695
Epoch 07 | train loss: 0.6057 | train acc: 0.7978 | val loss: 1.3541 | val acc: 0.5582
Epoch 08 | train loss: 0.5582 | train acc: 0.8144 | val loss: 1.3318 | val acc: 0.5709
Epoch 09 | train loss: 0.5172 | train acc: 0.8300 | val loss: 1.3543 | val acc: 0.5735
Epoch 10 | train loss: 0.5312 | train acc: 0.8310 | val loss: 1.4279 | val acc: 0.5549
Training done! Time taken: 11.763 seconds
bigrams-and-trigrams
Vocabulary size: 58541
Transforming documents...


In [ ]:
print(encoder_accuracy)

{'token-indicator': np.float64(0.5912727736017327), 'bag-of-words': np.float64(0.5878138227091915), 'bigrams-and-trigrams': np.float64(0.594600161379369)}


In [ ]:
import analysis

# First, a little helper, to gather all accuracies across runs and folds
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

posterior = analysis.BenavoliCorrelatedT(
    get_all_measurements_for(results, 'token-indicator').flatten(),  # this will have shape [num_runs * num_folds]
    get_all_measurements_for(results, 'bag-of-words').flatten(),  # this will have shape [num_runs * num_folds]
    names=["mlp2-indicator-enc", "mlp2-bow"]
)

p_a, p_tie, p_b = posterior.probs(0.005)
print(f"A=logistic_indicator-enc B=mlp_indicator-enc: P(A wins)={p_a:.4f} P(Tie)={p_tie:.4f} P(B wins)={p_b:.4f} Effect Size={posterior.effect_size:.2f}")
_ = posterior.plot(0.005) 